# Cost & latency bench

Quality metrics tell you *which* model is better. Cost and latency tell you *whether* the better one is worth deploying. This bench runs the same prompt through a 4-model panel and reports per-query cost and latency side-by-side.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
leaf = pathlib.Path.cwd()
sys.path.insert(0, str(leaf))
from eval import PANEL, _count_tokens
for model, p_in, p_out, _ in PANEL:
    print(f'{model:<45}  in=${p_in:.2f}/M  out=${p_out:.2f}/M')

## Token accounting is deterministic

`tiktoken` with the `cl100k_base` encoder gives a stable token count regardless of which provider you actually called. That's important — without a fixed encoder, token counts and therefore *costs* would jitter with every CI run.

In [ ]:
for s in ('Hello, world!',
          'By how much does RA-MoE reduce p99 decode latency?',
          'A long-ish system prompt about being precise and citing sources by bracketed id.'):
    print(f'{_count_tokens(s):3d}  | {s}')

## Run the bench

In [ ]:
import json, pathlib
snap = json.loads(pathlib.Path('eval-snapshot.json').read_text())
print(f'{"model":<45}  {"$/Q":>10}  {"ms/Q":>8}  {"in tok":>8}  {"out tok":>8}  pareto')
for m, v in snap['metrics']['per_model'].items():
    print(f"{m:<45}  {v['cost_per_query_usd']:>10.6f}  {v['avg_latency_ms']:>8.1f}  {v['avg_input_tokens']:>8.1f}  {v['avg_output_tokens']:>8.1f}  {v['pareto']}")

## Reading the pareto column

* **non-dominated** — no other panel model wins on *both* cost and latency. These are the candidates worth quality-testing.
* **dominated** — at least one other model wins on both axes. You shouldn't pick a dominated model unless you have a quality reason that the bench can't see (e.g., "this model is the only one that follows the system prompt under jailbreaks").

## What you can extend

* Replace the static `PANEL` price table with [LiteLLM's `model_prices_and_context_window.json`](https://github.com/BerriAI/litellm/blob/main/model_prices_and_context_window.json) for always-fresh pricing.
* Run the bench against a *quality* metric (faithfulness / answer_relevancy from `00-ragas-rag-eval/`) and plot the 3D pareto front: cost × latency × quality.
* Include p50 / p95 / p99 latency by running each query 5–10× (this notebook reports the mean only).